# Micky Assistant — Montage Pipeline

تجهيز البيئة وتشغيل خط المونتاج الكامل (P4-T1 → P4-T6) على Kaggle GPU.

الخطوات:
1. تثبيت الاعتماديات (Whisper + ffmpeg + moviepy + Remotion CLI)
2. استنساخ مستودع `micky-infra` ونسخ مصدر المونتاج
3. تنزيل الأصول (LUTs + خلفيات + خطوط عربية)
4. قراءة معطيات المهمة من متغيرات البيئة
5. تشغيل `pipeline.run_pipeline` الكامل
6. رفع المخرجات (5 منصات + captions) إلى GitHub Release

In [ ]:
# Cell 2 — تثبيت الاعتماديات
import subprocess, sys

def run(cmd, check=True):
    print(f'$ {cmd}')
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(res.stdout)
    if res.stderr:
        print(res.stderr, file=sys.stderr)
    if check and res.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')
    return res

# Python packages for the montage stack
!pip install -q openai-whisper torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q ffmpeg-python moviepy Pillow opencv-python
!pip install -q deepfilternet transformers accelerate

# System: ffmpeg + Node (for Remotion React renderer)
!apt-get update -qq && apt-get install -y -qq ffmpeg
!npm install -g npx
# Remotion CLI (used by remotion_renderer.py)
!npx -y create-video@latest --no-open --no-install /tmp/remotion-init >/dev/null 2>&1 || true
!npm install -g @remotion/cli@4

# Kaggle API (for downloading private dataset assets)
!pip install -q kaggle

print('\n[OK] Dependencies installed.')

In [ ]:
# Cell 3 — استنساخ المستودع + تنزيل الأصول
import os, shutil, subprocess

REPO_URL = 'https://github.com/mickystudio/micky-infra.git'
REPO_DIR = '/kaggle/working/micky-infra'
MONTAGE_SRC_DIR = '/kaggle/working/montage'
ASSETS_DIR = '/kaggle/input/micky-assets'

# Clone (use token if present, else public)
gh_token = os.environ.get('GH_TOKEN', '')
clone_url = REPO_URL
if gh_token:
    clone_url = REPO_URL.replace('https://', f'https://x-access-token:{gh_token}@')

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 {clone_url} {REPO_DIR}

# Copy montage python source into a stable path
if os.path.exists(MONTAGE_SRC_DIR):
    shutil.rmtree(MONTAGE_SRC_DIR)
shutil.copytree(os.path.join(REPO_DIR, 'kaggle-src', 'montage'), MONTAGE_SRC_DIR)
print(f'[OK] Montage source copied to {MONTAGE_SRC_DIR}')

# Download private dataset (LUTs, backgrounds, fonts)
if not os.path.exists(ASSETS_DIR):
    os.makedirs(ASSETS_DIR, exist_ok=True)
    try:
        !kaggle datasets download -d micky151246/micky-assets -p {ASSETS_DIR} --unzip
        print(f'[OK] Assets downloaded to {ASSETS_DIR}')
    except Exception as e:
        print(f'[WARN] Could not download assets dataset: {e}')
        print('      Falling back to repo-bundled assets if available.')
else:
    print(f'[OK] Assets already present at {ASSETS_DIR}')

# Quick sanity check
for mod in ('pipeline.py', 'audio_enhancer.py', 'background_remover.py',
            'take_selector.py', 'color_grader.py', 'caption_generator.py',
            'remotion_renderer.py', 'multi_export.py'):
    p = os.path.join(MONTAGE_SRC_DIR, mod)
    print(f'  {mod:30s} {"OK" if os.path.exists(p) else "MISSING"}')

In [ ]:
# Cell 4 — قراءة معطيات المهمة
import os, json

task_id     = os.environ.get('KAGGLE_TASK_ID', 'test')
input_video = os.environ.get('INPUT_VIDEO', '/kaggle/input/test_video.mp4')
options_str = os.environ.get('TASK_OPTIONS', '{}')
try:
    options = json.loads(options_str)
except json.JSONDecodeError as e:
    print(f'[WARN] TASK_OPTIONS JSON invalid ({e}); defaulting to {{}}')
    options = {}

# Defaults aligned with plan P4-T7 ready lists
options.setdefault('platforms', ['instagram', 'tiktok', 'youtube', 'facebook', 'whatsapp'])
options.setdefault('lut', 'medical_clean')            # P4-T4
options.setdefault('background', 'dental_office')     # P4-T3
options.setdefault('caption_style', 'hormozi')        # P4-T2
options.setdefault('denoise', True)                   # DeepFilterNet2
options.setdefault('loudness_target', -16)            # EBU R128

print(f'Task ID : {task_id}')
print(f'Input  : {input_video}')
print(f'Options: {json.dumps(options, ensure_ascii=False, indent=2)}')

if not os.path.exists(input_video):
    raise FileNotFoundError(f'Input video not found: {input_video}')
print(f'[OK] Input video exists ({os.path.getsize(input_video)/1e6:.1f} MB)')

In [ ]:
# Cell 5 — تشغيل خط المونتاج الكامل
import sys, json, time, traceback
sys.path.insert(0, '/kaggle/working/montage')

# Make assets discoverable for the modules that look them up
os.environ['MICKY_ASSETS_DIR'] = '/kaggle/input/micky-assets'
os.environ['MICKY_LUTS_DIR']    = '/kaggle/input/micky-assets/luts'
os.environ['MICKY_BGS_DIR']    = '/kaggle/input/micky-assets/backgrounds'
os.environ['MICKY_FONTS_DIR'] = '/kaggle/input/micky-assets/fonts'

from pipeline import run_pipeline  # kaggle-src/montage/pipeline.py

start = time.time()
try:
    result = run_pipeline(
        input_video_path=input_video,
        output_dir='/kaggle/working/output',
        options=options,
    )
    elapsed = time.time() - start
    print(f'\n[OK] Pipeline finished in {elapsed/60:.1f} min')
    print(json.dumps(result, ensure_ascii=False, indent=2))
except Exception as e:
    traceback.print_exc()
    raise RuntimeError(f'Pipeline failed: {e}')

In [ ]:
# Cell 6 — رفع النتائج إلى GitHub Release
# يتوقع أن `result` من الخلية السابقة يحتوي على 'outputs' (قائمة مسارات ملفات)
import os, glob

# Collect outputs explicitly + any extra files produced
outputs = result.get('outputs', []) if 'result' in globals() else []
if not outputs:
    # Fallback: grab everything in /kaggle/working/output
    outputs = sorted(glob.glob('/kaggle/working/output/**/*', recursive=True))
outputs = [p for p in outputs if os.path.isfile(p)]

print(f'Files to upload: {len(outputs)}')
for p in outputs:
    print(f'  - {p}  ({os.path.getsize(p)/1e6:.2f} MB)')

import os
import json
import subprocess
import urllib.request
import urllib.error

# --- إعدادات الرفع إلى GitHub Release ---
# يتم تمريرها عبر Kaggle Secrets أو متغيرات البيئة
GH_TOKEN = os.environ.get('GH_TOKEN', os.environ.get('GITHUB_TOKEN', ''))
GH_REPO = os.environ.get('GH_REPO', 'mickystudio/micky-infra')
TASK_ID = os.environ.get('KAGGLE_TASK_ID', 'test')
RELEASE_TAG = os.environ.get('RELEASE_TAG', f'task-{TASK_ID}')


def gh_release_api(method: str, path: str, payload: dict | None = None,
                   upload_url: str | None = None, asset_path: str | None = None):
    """Calls the GitHub REST API (or uploads an asset) with auth."""
    headers = {
        'Accept': 'application/vnd.github+json',
        'Authorization': f'Bearer {GH_TOKEN}',
        'X-GitHub-Api-Version': '2022-11-28',
    }
    try:
        if upload_url and asset_path:
            # Upload asset via multipart form
            import mimetypes
            mime = mimetypes.guess_type(asset_path)[0] or 'application/octet-stream'
            headers['Content-Type'] = mime
            with open(asset_path, 'rb') as fh:
                req = urllib.request.Request(upload_url, data=fh.read(), headers=headers, method='POST')
        else:
            data = json.dumps(payload).encode() if payload else None
            base = 'https://api.github.com'
            url = f'{base}{path}' if upload_url is None else upload_url
            req = urllib.request.Request(url, data=data, headers=headers, method=method)
        with urllib.request.urlopen(req, timeout=120) as resp:
            body = resp.read().decode('utf-8', errors='ignore')
            return resp.status, json.loads(body) if body else {}
    except urllib.error.HTTPError as e:
        err_body = e.read().decode('utf-8', errors='ignore')
        print(f'GitHub API error {e.code}: {err_body}')
        return e.code, {}
    except Exception as e:
        print(f'GitHub upload exception: {e}')
        return 0, {}


def ensure_release():
    """Create the release if it doesn't exist, return its upload_url template."""
    # Try fetching first
    status, data = gh_release_api('GET', f'/repos/{GH_REPO}/releases/tags/{RELEASE_TAG}')
    if status == 200:
        upload_url = data.get('upload_url', '').replace('{?name,label}', '')
        return upload_url
    # Create the release
    payload = {
        'tag_name': RELEASE_TAG,
        'name': f'Kaggle task {TASK_ID}',
        'body': 'Auto-uploaded from Kaggle notebook',
        'draft': False,
        'prerelease': False,
    }
    status, data = gh_release_api('POST', f'/repos/{GH_REPO}/releases', payload=payload)
    return data.get('upload_url', '').replace('{?name,label}', '') if data else ''


def upload_asset(upload_url: str, asset_path: str) -> str | None:
    """Uploads a single file to the release. Returns the browser_download_url."""
    name = os.path.basename(asset_path)
    url = f'{upload_url}?name={name}'
    status, data = gh_release_api('POST', '', upload_url=url, asset_path=asset_path)
    if status in (201, 200) and data:
        print(f'  uploaded: {name} -> {data.get("browser_download_url")}')
        return data.get('browser_download_url')
    print(f'  FAILED to upload {name} (status={status})')
    return None


def upload_files_to_release(file_paths, manifest=None):
    """Uploads all files to the GitHub release for this task."""
    if not GH_TOKEN:
        print('WARNING: GH_TOKEN not set. Skipping upload to GitHub Release.')
        print('Files remain available on the Kaggle output at /kaggle/working/output')
        return []
    upload_url = ensure_release()
    if not upload_url:
        print('ERROR: could not create/find GitHub release. Aborting upload.')
        return []
    uploaded = []
    for fp in file_paths:
        if not os.path.exists(fp):
            print(f'  skip (missing): {fp}')
            continue
        url = upload_asset(upload_url, fp)
        if url:
            uploaded.append(url)
    # Optional manifest (JSON describing outputs)
    if manifest:
        manifest_path = '/kaggle/working/manifest.json'
        with open(manifest_path, 'w') as f:
            json.dump(manifest, f, ensure_ascii=False, indent=2)
        upload_asset(upload_url, manifest_path)
    return uploaded


print(f'Release tag: {RELEASE_TAG}')
print(f'Repo:         {GH_REPO}')
print(f'Token set:    {bool(GH_TOKEN)}')

# Build manifest with task metadata + outputs
manifest = {
    'task_id': task_id,
    'input_video': input_video,
    'options': options,
    'outputs': [{'file': os.path.basename(p), 'size_mb': round(os.path.getsize(p)/1e6, 2)} for p in outputs],
}
uploaded_urls = upload_files_to_release(outputs, manifest=manifest)
print(f'\n[OK] {len(uploaded_urls)} files uploaded to GitHub Release.')
for u in uploaded_urls:
    print(f'  {u}')

## تم! النتائج جاهزة

الآن يمكن للـ Worker استطلاع حالة الـ Release وإرسال روابط التنزيل عبر تليجرام.
Kaggle سيتوقف تلقائيًا بعد انتهاء الجلسة (12h/جلسة) — تأكد من `kaggle_runner.ts` يوقف النواة بعد استلام الـ manifest.